# Inspect and compare

> **Install for this page:** `pip install scope-profiler` (base install only).

`scope-profiler inspect` summarises one profile and `scope-profiler diff`
compares two (or two sets). Both take `--top` to keep the output short and
`--json` for a structured result that scripts and agents can read. This page
also shows `scope-profiler tui` for browsing a file interactively.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# Work in a scratch directory so nothing lands in the working directory,
# and make the `scope-profiler` command of this environment available.
WORKDIR = Path(tempfile.mkdtemp(prefix="scope-profiler-guide-"))
os.chdir(WORKDIR)
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
os.environ["PYTHONPATH"] = os.pathsep.join(
    filter(None, [str(WORKDIR), os.environ.get("PYTHONPATH")])
)
sys.path.insert(0, str(WORKDIR))

In [ ]:
%%writefile app.py
import os
import time

import scope_profiler as sp

# SLOW scales the solver, to give the comparison something to find.
SLOW = float(os.environ.get("SLOW", "1"))

for _ in range(5):
    with sp.region("step"):
        with sp.region("assemble"):
            time.sleep(0.001)
        with sp.region("solve"):
            time.sleep(0.03 * SLOW)
        with sp.region("write"):
            time.sleep(0.0005)

In [ ]:
%%bash
for i in 1 2 3; do
    scope-profiler run -q -o base_$i.h5 app.py
    SLOW=3 scope-profiler run -q -o cand_$i.h5 app.py
done

## Inspect

`--sort total --top N` prints the N most expensive regions; `--regions-only`
drops the metadata block.

In [ ]:
%%bash
scope-profiler inspect base_1.h5 --regions-only --sort total --top 3

`--json` prints the same summary as JSON (one object for one file, a list for
several files or a glob):

In [ ]:
import json
import subprocess

out = subprocess.run(
    [
        "scope-profiler",
        "inspect",
        "base_1.h5",
        "--sort",
        "total",
        "--top",
        "2",
        "--json",
    ],
    capture_output=True,
    text=True,
    check=True,
).stdout
summary = json.loads(out)
list(summary)

## Diff

`diff` compares the region statistics of two files. The "Overall" line gives
the speedup of the whole run, from setup to finalize, so in a very short run it
also picks up the start-up and the writing of the file. `--top` keeps the largest changes, `--threshold`
hides changes below a percentage.

In [ ]:
%%bash
scope-profiler diff base_1.h5 cand_1.h5 --top 3

Timings are noisy. Before claiming a speedup of a few percent, profile several
runs per side and compare the globs (quote them so the shell does not expand
them); changes within the run-to-run spread are marked `~`.

In [ ]:
%%bash
scope-profiler diff 'base_*.h5' 'cand_*.h5' --top 3

In [ ]:
out = subprocess.run(
    ["scope-profiler", "diff", "base_1.h5", "cand_1.h5", "--json", "--threshold", "20"],
    capture_output=True,
    text=True,
    check=True,
).stdout
comparison = json.loads(out)
list(comparison)

## Browse with the TUI

`scope-profiler tui profile.h5` opens an interactive terminal browser of the
regions, their statistics and plots (it needs a terminal, so it is not run on
this page). Plots saved from it go to `--plot-output`, by default
`<file stem>_plots` next to the file.

```bash
scope-profiler tui base_1.h5
```